# PIVOT, UNPIVOT i dynamiczny SQL — notatki referencyjne (SQL Server)

Notatki do powrotu w codziennej pracy. Przykłady na uproszczonym modelu: `dim_baza_kadrowa` (numer osobowy, nazwa oddziału, stanowisko, data bazy) oraz `fact_Sprzedaz` (oddział, miesiąc, kwota).

## 1. `PIVOT` — podstawy

### Składnia

```sql
SELECT <kolumny_grupujące>, [wartość1], [wartość2], ...
FROM (<zapytanie źródłowe>) AS ŹródłoDanych
PIVOT (
    <funkcja_agregująca>(<kolumna_wartości>)
    FOR <kolumna_do_rozbicia> IN ( [wartość1], [wartość2], ... )
) AS TabelaPivot
```

**Wymagane elementy:** funkcja agregująca (nawet jeśli logicznie oczekujesz jednej wartości na komórkę — SQL Server i tak wymaga agregacji), jawna lista wartości w `IN (...)` (statyczna, musisz je znać w momencie pisania zapytania).

### Przykład podstawowy — sprzedaż oddziałów w kolumnach miesięcy

Dane źródłowe (`fact_Sprzedaz`): wiersze `Oddzial, Miesiac, Kwota`. Chcemy: jeden wiersz na oddział, kolumny to miesiące.

```sql
SELECT Oddzial, [1] AS Styczen, [2] AS Luty, [3] AS Marzec
FROM (
    SELECT Oddzial, MONTH(DataSprzedazy) AS Miesiac, Kwota
    FROM fact_Sprzedaz
    WHERE YEAR(DataSprzedazy) = 2025
) AS Zrodlo
PIVOT (
    SUM(Kwota)
    FOR Miesiac IN ( [1], [2], [3] )
) AS Pvt;
```

**Dlaczego `SUM`, skoro spodziewasz się jednej wartości na kombinację oddział+miesiąc?** `PIVOT` **zawsze** wymaga funkcji agregującej — nawet jeśli wiesz, że dla danej kombinacji jest tylko jeden wiersz źródłowy. Jeśli chcesz "po prostu przenieść wartość" bez prawdziwej agregacji, a masz pewność co do unikalności, użyj `MAX`/`MIN` — dają ten sam wynik przy jednym wierszu, a semantycznie sygnalizują "biorę tę jedną wartość", nie "sumuję wiele".

### Pułapka — brakujące kombinacje dają `NULL`, nie `0`

Jeśli oddział nie miał sprzedaży w danym miesiącu, komórka wynikowa to `NULL`, nie `0`. Jeśli potrzebujesz zer:

```sql
SELECT Oddzial,
    ISNULL([1], 0) AS Styczen,
    ISNULL([2], 0) AS Luty,
    ISNULL([3], 0) AS Marzec
FROM ( ... ) AS Zrodlo
PIVOT ( SUM(Kwota) FOR Miesiac IN ( [1], [2], [3] ) ) AS Pvt;
```

## 2. `UNPIVOT` — odwrócenie operacji

### Składnia

```sql
SELECT <kolumny_zachowane>, <kolumna_nazwa>, <kolumna_wartość>
FROM <tabela_źródłowa>
UNPIVOT (
    <kolumna_wartość> FOR <kolumna_nazwa> IN ( [kolumna1], [kolumna2], ... )
) AS TabelaUnpivot
```

### Przykład — z powrotem z kolumn miesięcy do wierszy

```sql
SELECT Oddzial, Miesiac, Kwota
FROM (
    SELECT Oddzial, Styczen, Luty, Marzec
    FROM raport_sprzedazy_miesiace  -- tabela z kolumnami Styczen/Luty/Marzec
) AS Zrodlo
UNPIVOT (
    Kwota FOR Miesiac IN ( Styczen, Luty, Marzec )
) AS Unpvt;
```

Wynik: po 1 wierszu na każdą kombinację Oddział × Miesiąc, kolumna `Miesiac` zawiera tekst `'Styczen'`/`'Luty'`/`'Marzec'` (nazwę oryginalnej kolumny), `Kwota` — jej wartość.

### Pułapka — `UNPIVOT` domyślnie pomija wiersze z `NULL`

Jeśli `Luty` dla danego oddziału to `NULL`, ten wiersz **nie pojawi się w ogóle** w wyniku `UNPIVOT` (nie jako wiersz z `Kwota = NULL`, tylko w ogóle go nie będzie). Jeśli potrzebujesz zachować te wiersze:

```sql
SELECT Oddzial, Miesiac, Kwota
FROM (
    SELECT Oddzial,
        ISNULL(Styczen, 0) AS Styczen,
        ISNULL(Luty, 0) AS Luty,
        ISNULL(Marzec, 0) AS Marzec
    FROM raport_sprzedazy_miesiace
) AS Zrodlo
UNPIVOT ( Kwota FOR Miesiac IN ( Styczen, Luty, Marzec ) ) AS Unpvt;
```

Zamień `NULL` na `0` (albo inną wartość-strażnika) **przed** `UNPIVOT`, żeby wiersz przetrwał operację — trzeba świadomie zdecydować, czy zero jest tu poprawną interpretacją braku danych, czy zafałszowuje wynik (np. przy danych kadrowych zero mogłoby sugerować "zero godzin" zamiast "brak danych").

### Typy danych — muszą się zgadzać

`UNPIVOT` wymaga, żeby wszystkie kolumny wejściowe (`Styczen`, `Luty`, `Marzec`) miały **ten sam typ danych** (albo dawały się niejawnie skonwertować bez utraty danych) — inaczej błąd albo niechciana konwersja. Częsty problem przy mieszaniu `DECIMAL` o różnej precyzji.

## 3. Ograniczenie statycznego `PIVOT`/`UNPIVOT` — dlaczego w ogóle potrzebujesz dynamicznego SQL

Statyczny `PIVOT` wymaga wypisania **z góry, w kodzie SQL**, każdej wartości, która ma stać się kolumną (`[1], [2], [3]` w przykładzie wyżej). To działa dobrze, gdy zbiór wartości jest **stały i znany** (12 miesięcy, 4 kwartały). Nie działa, gdy zbiór jest **zmienny w czasie** — np. lista oddziałów, lista stanowisk, lista produktów — bo dziś masz 15 oddziałów, za pół roku 18, a zapytanie z zaszytą na sztywno listą 15 nazw nie uwzględni nowych bez ręcznej edycji kodu.

**Rozwiązanie: zbuduj listę kolumn i całe zapytanie `PIVOT` jako tekst w czasie wykonania (dynamiczny SQL), na podstawie aktualnej zawartości bazy.**

## 4. Dynamiczny `PIVOT` — pełny wzorzec krok po kroku

### Krok 1 — zbuduj listę unikalnych wartości jako string, przez `STRING_AGG`

```sql
DECLARE @Kolumny NVARCHAR(MAX);

SELECT @Kolumny = STRING_AGG(QUOTENAME(Oddzial), ', ')
FROM (SELECT DISTINCT Oddzial FROM fact_Sprzedaz) AS UnikalneOddzialy;
```

**`QUOTENAME(Oddzial)`** — kluczowe, nie pomijaj. Otacza każdą nazwę nawiasami kwadratowymi (`[Warszawa Centrum]`) — **to jest podstawowe zabezpieczenie przed SQL injection** w tym konkretnym miejscu (nazwa oddziału, która trafia do finalnego tekstu zapytania, nie jako parametr, tylko jako fragment identyfikatora kolumny). Bez `QUOTENAME`, nazwa oddziału zawierająca np. `]; DROP TABLE ...` mogłaby (w teorii, przy nieostrożnej dalszej konstrukcji) wstrzyknąć dowolny kod SQL. `QUOTENAME` neutralizuje to, bo cokolwiek jest w środku, staje się literalną nazwą kolumny w nawiasach, nie wykonywalnym kodem.

**`STRING_AGG`** (SQL Server 2017+) — nowoczesny sposób łączenia wielu wierszy w jeden string z separatorem. Starsza alternatywa (kompatybilna wstecz) to sztuczka z `FOR XML PATH('')` + `STUFF` — pokazuję ją niżej dla kompletności, bo wciąż spotykana w starszym kodzie:

```sql
-- Starsza składnia, przed SQL Server 2017 (STRING_AGG niedostępne)
SELECT @Kolumny = STUFF((
    SELECT DISTINCT ', ' + QUOTENAME(Oddzial)
    FROM fact_Sprzedaz
    FOR XML PATH('')
), 1, 2, '');
```

### Krok 2 — zbuduj pełne zapytanie `PIVOT` jako tekst

```sql
DECLARE @SQL NVARCHAR(MAX);

SET @SQL = N'
SELECT Miesiac, ' + @Kolumny + N'
FROM (
    SELECT Oddzial, MONTH(DataSprzedazy) AS Miesiac, Kwota
    FROM fact_Sprzedaz
    WHERE YEAR(DataSprzedazy) = 2025
) AS Zrodlo
PIVOT (
    SUM(Kwota)
    FOR Oddzial IN (' + @Kolumny + N')
) AS Pvt
ORDER BY Miesiac;';
```

### Krok 3 — wykonaj przez `sp_executesql`, NIE przez `EXEC(@SQL)`

```sql
EXEC sp_executesql @SQL;
```

**Dlaczego `sp_executesql`, a nie zwykły `EXEC(@SQL)`:** `sp_executesql` pozwala **parametryzować** części zapytania, które są prawdziwymi wartościami danych (nie nazwami kolumn) — np. rok, zakres dat, próg filtru. `EXEC(@SQL)` wykonuje czysty tekst bez żadnej możliwości parametryzacji, co wymusza wklejanie **wszystkich** wartości wprost do stringa (większe ryzyko SQL injection przy wartościach pochodzących od użytkownika, gorszy plan cache'owania zapytań w SQL Server, bo każda kombinacja tekstu to inny plan).

**Wersja z parametrem (rok jako parametr, nie wklejony na sztywno):**

```sql
DECLARE @SQL NVARCHAR(MAX);
DECLARE @Rok INT = 2025;

SET @SQL = N'
SELECT Miesiac, ' + @Kolumny + N'
FROM (
    SELECT Oddzial, MONTH(DataSprzedazy) AS Miesiac, Kwota
    FROM fact_Sprzedaz
    WHERE YEAR(DataSprzedazy) = @RokParam
) AS Zrodlo
PIVOT ( SUM(Kwota) FOR Oddzial IN (' + @Kolumny + N') ) AS Pvt
ORDER BY Miesiac;';

EXEC sp_executesql @SQL, N'@RokParam INT', @RokParam = @Rok;
```

`@Kolumny` (nazwy kolumn) **musi** trafić do tekstu zapytania bezpośrednio — nazw kolumn nie da się parametryzować przez `sp_executesql` (to nie są wartości, to część struktury zapytania). `@Rok` (prawdziwa wartość danych) **powinien** być parametrem — to jest granica, którą warto rozumieć: struktura zapytania (kolumny, nazwy tabel) = budowana tekstowo z `QUOTENAME`; wartości danych = zawsze parametryzowane.

## 5. Dynamiczny `UNPIVOT` — analogiczny wzorzec

Przydatny, gdy masz tabelę z nieznaną z góry liczbą kolumn "szerokich" (np. eksport z arkusza, gdzie kolumny to nazwy miesięcy/produktów) i chcesz je znormalizować do formatu wierszowego.

```sql
DECLARE @KolumnyUnpivot NVARCHAR(MAX);
DECLARE @SQL NVARCHAR(MAX);

-- Krok 1: lista kolumn do rozpisania w wiersze (wszystkie poza kolumnami "kluczowymi")
SELECT @KolumnyUnpivot = STRING_AGG(QUOTENAME(COLUMN_NAME), ', ')
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_NAME = 'raport_sprzedazy_miesiace'
  AND COLUMN_NAME NOT IN ('Oddzial');   -- kolumny "kluczowe", które zostają jako są

-- Krok 2: budowa zapytania UNPIVOT
SET @SQL = N'
SELECT Oddzial, Miesiac, Kwota
FROM raport_sprzedazy_miesiace
UNPIVOT ( Kwota FOR Miesiac IN (' + @KolumnyUnpivot + N') ) AS Unpvt;';

EXEC sp_executesql @SQL;
```

**Źródło listy kolumn: `INFORMATION_SCHEMA.COLUMNS`** — metadane systemowe SQL Server, opisujące strukturę tabeli w danym momencie. To pozwala zapytaniu **samo się dostosować**, gdy ktoś doda nową kolumnę-miesiąc do tabeli źródłowej, bez zmiany kodu.

## 6. Alternatywa bez `PIVOT`/`UNPIVOT` — ręczny wzorzec `CASE WHEN` + `GROUP BY`

Warto znać tę alternatywę, bo bywa **czytelniejsza i czasem wydajniejsza** przy prostych przypadkach, szczególnie gdy liczba kolumn docelowych jest mała i znana:

```sql
-- Ręczny odpowiednik statycznego PIVOT z sekcji 1
SELECT
    Oddzial,
    SUM(CASE WHEN MONTH(DataSprzedazy) = 1 THEN Kwota ELSE 0 END) AS Styczen,
    SUM(CASE WHEN MONTH(DataSprzedazy) = 2 THEN Kwota ELSE 0 END) AS Luty,
    SUM(CASE WHEN MONTH(DataSprzedazy) = 3 THEN Kwota ELSE 0 END) AS Marzec
FROM fact_Sprzedaz
WHERE YEAR(DataSprzedazy) = 2025
GROUP BY Oddzial;
```

**Kiedy wybrać `CASE WHEN`+`GROUP BY` zamiast `PIVOT`:**
- Chcesz uniknąć `NULL` dla brakujących kombinacji (tu masz `0` "za darmo", z `ELSE 0`, bez osobnego `ISNULL`).
- Liczba kolumn docelowych jest mała, znana i rzadko się zmienia (miesiące, kwartały) — nie potrzebujesz dynamicznego SQL wcale.
- Zespół, który utrzymuje kod, lepiej zna `CASE WHEN`/`GROUP BY` niż specyficzną składnię `PIVOT` — to reguła, którą się kieruje wiele zespołów SQL: `PIVOT` bywa mniej intuicyjny dla osób spoza świata SQL Server (nie każdy silnik SQL go ma — MySQL/PostgreSQL wymagają wprost tego ręcznego wzorca, bo nie mają natywnego `PIVOT`).

**Kiedy `PIVOT` ma przewagę:** duża, zmienna liczba kolumn wynikowych (wtedy i tak lądujesz przy dynamicznym SQL niezależnie od podejścia — a natywny `PIVOT` bywa czytelniejszy w dynamicznie budowanym tekście niż odpowiednik z wieloma `SUM(CASE WHEN ...)` sklejanymi w pętli).

## 7. Podsumowanie — ściąga

| Potrzebujesz | Rozwiązanie |
|---|---|
| Znana z góry, stała lista kolumn docelowych (np. 12 miesięcy) | Statyczny `PIVOT` albo `CASE WHEN`+`GROUP BY` — dowolnie, kwestia stylu |
| Nieznana, zmienna w czasie lista kolumn (oddziały, produkty) | Dynamiczny SQL: `STRING_AGG`+`QUOTENAME` do budowy listy, `sp_executesql` do wykonania |
| Zamiana kolumn z powrotem na wiersze, stała lista kolumn | Statyczny `UNPIVOT` |
| Zamiana kolumn na wiersze, nieznana lista kolumn | Dynamiczny `UNPIVOT` z `INFORMATION_SCHEMA.COLUMNS` |
| Wartości danych do wstrzyknięcia w zapytanie dynamiczne | Zawsze przez parametr `sp_executesql`, nigdy wklejone bezpośrednio do stringa |
| Nazwy kolumn/tabel do wstrzyknięcia w zapytanie dynamiczne | Zawsze przez `QUOTENAME`, wklejone do tekstu (nie da się inaczej — to struktura, nie wartość) |
| Chcesz uniknąć `NULL` w wyniku `PIVOT` | `ISNULL(...)` na każdej kolumnie wynikowej, albo `CASE WHEN`+`ELSE 0` od razu |
| `UNPIVOT` "gubi" wiersze z `NULL` w kolumnach źródłowych | `ISNULL(...)` na kolumnach źródłowych przed `UNPIVOT` |